# Real Estate Price Estimator - EDA and Training

This notebook follows the required pipeline structure from data loading to final model export.

## 1. Import Libraries

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/housing_clean.csv')
df.head()

## 3. Initial Inspection

In [ ]:
print('Shape:', df.shape)
display(df.dtypes)
display(df.isna().sum())
display(df.describe().T)

## 4. Data Cleaning

Cleaning is performed by `src/cleaning.py`; we inspect results only.

In [ ]:
with open('../model/training_metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)
metadata['cleaning_summary']

## 5. Exploratory Data Analysis

In [ ]:
num_cols = ['price', 'sqft', 'bedrooms', 'bathrooms', 'age']
df[num_cols].hist(figsize=(12, 8), bins=30)
plt.tight_layout()

In [ ]:
corr = df[num_cols].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.scatterplot(data=df, x='sqft', y='price', ax=axes[0,0])
sns.scatterplot(data=df, x='bedrooms', y='price', ax=axes[0,1])
sns.scatterplot(data=df, x='bathrooms', y='price', ax=axes[1,0])
sns.scatterplot(data=df, x='age', y='price', ax=axes[1,1])
plt.tight_layout()

In [ ]:
plt.figure(figsize=(12, 6))
for i, c in enumerate(num_cols, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(y=df[c])
    plt.title(c)
plt.tight_layout()

In [ ]:
sns.pairplot(df[num_cols], corner=True)

## 6. Feature Engineering

In [ ]:
features = ['sqft', 'bedrooms', 'bathrooms', 'age']
X = df[features].copy()
y = df['price'].copy()
price_skew = y.skew()
use_log = abs(price_skew) > 1.0
if use_log:
    y = np.log1p(y)
print({'price_skew': price_skew, 'used_log_price': use_log})

## 7. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## 8. Model Training

In [ ]:
models = {
    'LinearRegression': LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1),
    'GradientBoostingRegressor': GradientBoostingRegressor(random_state=42),
}
trained = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    trained[name] = model

## 9. Model Evaluation

In [ ]:
results = {}
for name, model in trained.items():
    pred = model.predict(X_test_s)
    if use_log:
        y_true_eval = np.expm1(y_test)
        pred_eval = np.expm1(pred)
    else:
        y_true_eval = y_test
        pred_eval = pred
    results[name] = {
        'RMSE': float(np.sqrt(mean_squared_error(y_true_eval, pred_eval))),
        'MAE': float(mean_absolute_error(y_true_eval, pred_eval)),
        'R2': float(r2_score(y_true_eval, pred_eval)),
    }
pd.DataFrame(results).T.sort_values('RMSE')

## 10. Model Comparison

In [ ]:
best_model_name = min(results, key=lambda m: results[m]['RMSE'])
best_model_name, results[best_model_name]

## 11. Save Final Model

In [ ]:
best_model = trained[best_model_name]
joblib.dump(best_model, '../model/model.pkl')
joblib.dump(scaler, '../model/scaler.pkl')
with open('../model/feature_order.json', 'w', encoding='utf-8') as f:
    json.dump(features, f, indent=2)

## 12. Conclusions

- The pipeline uses 4 interpretable predictors and reproducible preprocessing.
- Multiple models are compared with RMSE, MAE, and R2.
- Final artifacts are stored in `model/` for backend inference.